---
<div style='background-color:honeydew; font-size: 24px;'>
    <center><strong>Encoding Documentation</strong></center>
</div>

---
The is the final documentation notebook, `encoding`, will walk you through the iris recognition pipeline beyond segmentation, covering normalization, quality assessment, feature extraction, and encoding through:

- Converting the circular iris region into a normalized rectangular strip using LinearNormalization.
- Assessing image focus and sharpness with SharpnessEstimation and the SharpnessValidator.
- Extract complex Gabor filter responses from the normalized iris using ConvFilterBank.
- Identify and mask fragile/unreliable bits in the iris code with FragileBitRefinement.
- Encode the Gabor responses into a binary iris template using IrisEncoder and validate usable bits with IsMaskTooSmallValidator.
- Compute the bounding box of the iris in the original image with IrisBBoxCalculator.
- Visualize intermediate and final outputs, including normalized strips, noise masks, filter responses, refined masks, and binary iris codes.

---
<div style='background-color:honeydew; font-size: 24px;'>
    <center><strong>LinearNormalization Node</strong></center>
</div>

---

Example from pipeline.yaml:

<code> - <font color='green'>name: normalization</font>
    algorithm:    
      class_name: <font color='blue'>iris.LinearNormalization</font>
      params: {}</code> If this is empty, the default values are used.
      
<code>    inputs:
      - name: image
        source_node: input
      - name: noise_mask
        source_node: noise_masks_aggregation
      - name: extrapolated_contours
        source_node: geometry_estimation
      - name: eye_orientation
        source_node: eye_orientation
    callbacks:</code>

### What it does

LinearNormalization converts the iris from its natural circular form in the original image into a flat rectangular strip. It was originally mentioned in the Daugman paper called [How Iris Recognition Works](https://www.robots.ox.ac.uk/~az/lectures/est/iris.pdf). To do so, pixels are sampled along evenly spaced radial lines between the pupil and iris boundaries, basically "unrolling" the iris. It makes the iris texture consistent regardless of pupil dilation or eye rotation. 

**Key Parameters**


| Parameter | Input Type | Default Value | What it controls |
|---|---|---|---|
| `res_in_r` | `PositiveInt` | 128 | Height of the output strip - how many radial sampels to take between pupil and iris boundary. Higher values represent more detail in the radial direction |
| `oversat_threshold` | `PositiveInt` | 254 | Pixel brightness value above which a pixel is considered oversaturated and treated as noise. Maximum value is 255. |


**Code \& Visualization Cells**


In [ ]:
# ---- Imports ----
import cv2
import iris
from pathlib import Path
from iris.nodes.normalization.linear_normalization import LinearNormalization

# ---- Load image ----
notebook_dir = Path.cwd() # Get the current working directory (the folder where the notebook is running)

# Go up one level to the colab folder
img_path = notebook_dir.parent / "sample_ir_image.png"

img_pixels = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)

ir_image = iris.IRImage(img_data=img_pixels, image_id="sample", eye_side="left")

# ---- Run full pipeline (populates call_trace) ----
pipeline = iris.IRISPipeline()
output = pipeline(ir_image)

# ---- Pull intermediate outputs from call_trace ----
noise_mask = pipeline.call_trace.get("noise_masks_aggregation")
geometry   = pipeline.call_trace.get("geometry_estimation")
eye_orient = pipeline.call_trace.get("eye_orientation")

print("noise_mask type:", type(noise_mask))
print("geometry type:  ", type(geometry))
print("eye_orient type:", type(eye_orient))

# ---- Run LinearNormalization in isolation ----
node = LinearNormalization(res_in_r=128, oversat_threshold=254)

result = node.run(
    image=ir_image,
    noise_mask=noise_mask,
    extrapolated_contours=geometry,
    eye_orientation=eye_orient
)

print("\nResult type:", type(result))
print("Normalized image shape:", result.normalized_image.shape)
print("Normalized mask shape: ", result.normalized_mask.shape)

In [ ]:
# ---- Imports ----
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Top left: original eye image with iris/pupil boundaries overlaid
axes[0, 0].imshow(img_pixels, cmap="gray")
iris_pts = geometry.iris_array
pupil_pts = geometry.pupil_array
axes[0, 0].plot(iris_pts[:, 0], iris_pts[:, 1], 
                "cyan", linewidth=1.5, label="Iris boundary")
axes[0, 0].plot(pupil_pts[:, 0], pupil_pts[:, 1], 
                "yellow", linewidth=1.5, label="Pupil boundary")
axes[0, 0].legend(loc="upper right", fontsize=8)
axes[0, 0].set_title("Input: Original Eye Image\n(with fitted boundaries)")
axes[0, 0].axis("off")

# Top right: normalized iris strip
axes[0, 1].imshow(result.normalized_image, cmap="gray", aspect="auto")
axes[0, 1].set_title("Output: Normalized Iris Strip\n(Daugman rubber sheet)")
axes[0, 1].set_xlabel("Angular position (around iris) →")
axes[0, 1].set_ylabel("Radial position →")

# Bottom left: noise mask overlaid on original image
noise_display = np.zeros((*img_pixels.shape, 3), dtype=np.uint8)
noise_display[..., 0] = noise_mask.mask.astype(np.uint8) * 255
axes[1, 0].imshow(img_pixels, cmap="gray")
axes[1, 0].imshow(noise_display, alpha=0.4)
axes[1, 0].set_title("Input: Noise Mask\n(red = noisy/occluded regions)")
axes[1, 0].axis("off")

# Bottom right: normalized mask
axes[1, 1].imshow(result.normalized_mask, cmap="RdYlGn", aspect="auto")
axes[1, 1].set_title("Output: Normalized Noise Mask\n(green = reliable, red = masked out)")
axes[1, 1].set_xlabel("Angular position →")
axes[1, 1].set_ylabel("Radial position →")

plt.suptitle("LinearNormalization: Iris Unrolling", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# Stats
print(f"Normalized strip dimensions: {result.normalized_image.shape} (height x width)")
print(f"Eye orientation angle: {eye_orient.angle:.4f} radians ({np.degrees(eye_orient.angle):.2f} degrees)")

### What to look for

The strip should look like a flattened ring of iris texture - you should see the radial fiber patterns of the iris running approximately vertically across the strip with relatively consistent brightness across the width. The noise mask should be mostly green/clean with red regions only at the top and bottom edges where eyelids occlude the iris.

| What you see | What it means|
|---|---|
|Large red/masked regions in the center of the strip|Heavy eyelid occlusion — this image may be rejected by the occlusion validators|
|Uniform gray strip with no visible texture|Iris boundary was fitted incorrectly — normalization sampled outside the iris|
|Bright horizontal bands|Specular reflections weren't fully masked, corrupting the texture|
|Strip looks "stretched" on one side|Eye was off-gaze, causing uneven sampling around the boundary|
|Very narrow strip|res_in_r is low, or the pupil and iris boundaries are very close together


---

<div style='background-color:honeydew; font-size: 24px;'>
    <center><strong>SharpnessEstimation Node</strong></center>
</div>

---
  
Example from pipeline.yaml:

<code>   - <font color='green'>name: sharpness_estimation</font>
    algorithm:
      class_name: <font color='blue'>iris.SharpnessEstimation</font>
      params: {}
    inputs:
      - name: normalization_output
        source_node: normalization
    callbacks:
      - class_name: <font color='blue'>iris.nodes.validators.object_validators.SharpnessValidator</font>
        params:
          min_sharpness: 461.0</code>

### What it does

SharpnessEstimation measures how in-focus the normalized iris strip is by computing the variance of the Laplacian, basically detecting how many sharp edges exist in the image. The SharpnessValidator callback then compares the score against a minimum threshold and rejects the image if it falls short.

**Key Parameters**  

| Parameter | Input Type | Default Value | What it controls |
|---|---|---|---|
| `SharpnessEstimation` | `none` | none | No configurable parameters - fixed algorithm |
| `min_sharpness` | `float` | 461.0 | Minimum Laplacian variance score to pass - image rejected for bluriness if below |



**Code & Visualization Cells**   



In [ ]:
from iris.nodes.eye_properties_estimation.sharpness_estimation import SharpnessEstimation

# Get the normalized iris from call_trace
normalization_output = pipeline.call_trace.get("normalization")

# Run sharpness estimation
node = SharpnessEstimation()
sharpness = node.run(normalization_output=normalization_output)

print(type(sharpness))
print("Sharpness score:", sharpness.score)
print("Passes threshold (>= 461.0):", sharpness.score >= 461.0)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Left: normalized iris strip
axes[0].imshow(normalization_output.normalized_image, cmap="gray", aspect="auto")
axes[0].set_title("Normalized Iris Strip")
axes[0].axis("off")

# Middle: Laplacian response (what sharpness is measuring)
laplacian = cv2.Laplacian(normalization_output.normalized_image.astype(np.float32), cv2.CV_32F)
axes[1].imshow(np.abs(laplacian), cmap="hot", aspect="auto")
axes[1].set_title("Laplacian Response\n(edges = bright = sharp)")
axes[1].axis("off")

# Right: score gauge
score = sharpness.score
threshold = 461.0
color = "green" if score >= threshold else "red"
axes[2].barh(["Sharpness"], [score], color=color)
axes[2].axvline(x=threshold, color="black", linestyle="--", label=f"Threshold ({threshold})")
axes[2].set_title(f"Sharpness Score: {score:.1f}\n({'PASS' if score >= threshold else 'FAIL'})")
axes[2].legend()

plt.suptitle("SharpnessEstimation", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


**What to look for**

A good image will have a score well above 461. A crips iris typically scores in the hundreds to thousands depending on the resolution. The Laplacian visualization should show string bright edges along the iris fiber patterns in a sharp image.

---

<div style='background-color:honeydew; font-size: 24px;'>
    <center><strong>Filter Bank Node</strong></center>
</div>

---
 
Example from pipeline.yaml:

<code>  - <font color='green'>name: filter_bank</font>
    algorithm:
      class_name: <font color='blue'>iris.ConvFilterBank</font>
      params:
        maskisduplicated: False
        filters:
          - class_name: <font color='blue'>iris.GaborFilter</font>
            params:
              kernel_size: [41, 21]
              sigma_phi: 7
              sigma_rho: 6.13
              theta_degrees: 90.0
              lambda_phi: 28.0
              dc_correction: True
              to_fixpoints: True
          - class_name: <font color='blue'>iris.GaborFilter</font>
            params:
              kernel_size: [17, 21]
              sigma_phi: 2
              sigma_rho: 5.86
              theta_degrees: 90.0
              lambda_phi: 8
              dc_correction: True
              to_fixpoints: True
        probe_schemas:
          - class_name: <font color='blue'>iris.RegularProbeSchema</font>
            params:
              n_rows: 16
              n_cols: 256
          - class_name: <font color='blue'>iris.RegularProbeSchema</font>
            params:
              n_rows: 16
              n_cols: 256
    inputs:
      - name: normalization_output
        source_node: normalization
    callbacks:</code>

### What it does 

The filter bank applies two Gabor filters at different scales to the normalized iris strip, sampling the responses at a 16x256 grid of probe points. A Gabor filter is a sine wave multiplied by a Gaussian envelope and it responds strongly to edges and texture at a specific frequency and orientation. The complex-valued response (real and imaginary) encodes the local phase of the iris texture, which becomes the iris code. 

**Key Parameters**  

| Parameter | Input Type | Default Value | What it controls |
|---|---|---|---|
| `kernel_size` | `Tuple[conint(gt=3, lt=99), conint(gt=3, lt=99)]` | None, user must provide | Size of the two Gabor kernels (larger = coarser features) |
| `sigma_phi` | `float` | None, user must provide | Angular spread of the Gaussian envelope |
| `sigma_rho` | `float` | None, user must provide | Radial spread of the Gaussian envelope |
| `lambda_phi` | `float ` | None, user must provide | Wavelength of the sinusoidal carrier - controls what frequency of texture is detected |
| `dc_correction` | `bool` | True | Removes mean intensity to make filter response lighting-invariant |
| `n_rows` | `int` | None, user must provide | Number of radial probe points to sample |
| `n_cols` | `int` | None, user must provide | Number of angular probe points to sample |

**Code & Visualization Cells**  

In [ ]:
from iris.nodes.iris_response.conv_filter_bank import ConvFilterBank
from iris.nodes.iris_response.image_filters.gabor_filters import GaborFilter
from iris.nodes.iris_response.probe_schemas.regular_probe_schema import RegularProbeSchema

normalization_output = pipeline.call_trace.get("normalization")

# Recreate the filter bank exactly as defined in the YAML
filter_bank = ConvFilterBank(
    maskisduplicated=False,
    filters=[
        GaborFilter(
            kernel_size=[41, 21],
            sigma_phi=7,
            sigma_rho=6.13,
            theta_degrees=90.0,
            lambda_phi=28.0,
            dc_correction=True,
            to_fixpoints=True
        ),
        GaborFilter(
            kernel_size=[17, 21],
            sigma_phi=2,
            sigma_rho=5.86,
            theta_degrees=90.0,
            lambda_phi=8,
            dc_correction=True,
            to_fixpoints=True
        )
    ],
    probe_schemas=[
        RegularProbeSchema(n_rows=16, n_cols=256),
        RegularProbeSchema(n_rows=16, n_cols=256)
    ]
)

response = filter_bank.run(normalization_output=normalization_output)

print(type(response))
print("Number of filter responses:", len(response.iris_responses))
print("Response shape (filter 1):", response.iris_responses[0].shape)
print("Response dtype:", response.iris_responses[0].dtype)  # should be complex

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 10))

# Top row: the two Gabor kernels themselves
for i, (sigma_phi, lambda_phi, label) in enumerate([
    (7, 28.0, "Filter 1 (coarse, λ=28)"),
    (2, 8.0,  "Filter 2 (fine, λ=8)")
]):
    # Show real part of response
    real_part = response.iris_responses[i].real
    axes[0, i].imshow(real_part, cmap="RdBu", aspect="auto")
    axes[0, i].set_title(f"{label}\nReal part of Gabor response")
    axes[0, i].set_xlabel("Angular position →")
    axes[0, i].set_ylabel("Radial position →")

# Middle row: imaginary parts
for i in range(2):
    imag_part = response.iris_responses[i].imag
    axes[1, i].imshow(imag_part, cmap="RdBu", aspect="auto")
    axes[1, i].set_title(f"Filter {i+1}: Imaginary part")
    axes[1, i].set_xlabel("Angular position →")

# Bottom row: phase (what actually gets encoded into the iris code)
for i in range(2):
    phase = np.angle(response.iris_responses[i])
    axes[2, i].imshow(phase, cmap="hsv", aspect="auto")
    axes[2, i].set_title(f"Filter {i+1}: Phase\n(this becomes the iris code bits)")
    axes[2, i].set_xlabel("Angular position →")

plt.suptitle("ConvFilterBank: Gabor Feature Extraction", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


### What to look for

The real and imaginary segments should show alternating bands of positive/negative response corresponding to the iris fiber texture. The phase plot should show relatively stable, structured regions of color rather than random noise. If the phase looks completely random with no structure, the iris texture was too blurry or noisy for reliable encoding. The coarse filter (filter 1) should show broader bands, while the fine filter (filter 2) shows finer-grained patterns.

---

<div style='background-color:honeydew; font-size: 24px;'>
    <center><strong>Iris Response Refinement</strong></center>
</div>

---

Example from pipeline.yaml:

<code>      - <font color='green'>name: iris_response_refinement</font>
    algorithm:
      class_name: <font color='blue'>iris.nodes.iris_response_refinement.fragile_bits_refinement.FragileBitRefinement</font>
      params:
        value_threshold:  [0.0001, 0.275, 0.08726646259971647]
        fragile_type: "polar"
        maskisduplicated: False
    inputs:
      - name: response
        source_node: filter_bank
    callbacks:</code>

### What it does  

fter the Gabor filter bank runs, some bits in the iris code are unreliable. At those points, the Gabor resonse magnitude is very close to zero, meaning the phase estimate there could flip with the tiniest change in the image. FragileBitRefinement indentifies these "fragile" bits by checking whether the response magnitude falls below certain thresholds, and adds them to the mask so they are ignored during matching. 

**Key Parameters** 

| Parameter | Input Type | Default Value | What it controls|
|---|---|---|---|
| `value_threshold` | `Tuple[confloat(ge=0), confloat(ge=0), confloat(ge=0)]` | None, user must provide | Three thresholds for identifying fragile bits - responses below these are masked |
| `fragile_type` | `FragileType` | FragileType.polar; param is also optional | Coordinate system used for the threshold check |
| `maskisduplicated` | `bool` | None, param is optional | Whether the mask is shared between filters or separate |

**Code & Visualization Cells**  





In [ ]:
from iris.nodes.iris_response_refinement.fragile_bits_refinement import FragileBitRefinement

response = pipeline.call_trace.get("filter_bank")

node = FragileBitRefinement(
    value_threshold=[0.0001, 0.275, 0.08726646259971647],
    fragile_type="polar",
    maskisduplicated=False
)

refined = node.run(response=response)

print(type(refined))
print("Number of responses:", len(refined.iris_responses))
print("iris_code_version:", refined.iris_code_version)

# Extract real part of masks (1.0 = valid, 0.0 = masked)
original_valid = response.mask_responses[0].real == 1
refined_valid  = refined.mask_responses[0].real == 1

fragile_bits = original_valid & ~refined_valid

print(f"\nBits valid before refinement: {original_valid.sum()}")
print(f"Bits valid after refinement:  {refined_valid.sum()}")
print(f"Fragile bits removed:         {fragile_bits.sum()}")
print(f"Percentage removed:           {100 * fragile_bits.sum() / original_valid.sum():.2f}%")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 6))

for i in range(2):
    # Response magnitude (low magnitude = fragile)
    magnitude = np.abs(refined.iris_responses[i])
    axes[i, 0].imshow(magnitude, cmap="viridis", aspect="auto")
    axes[i, 0].set_title(f"Filter {i+1}: Response Magnitude\n(dark = near zero = fragile)")
    axes[i, 0].set_xlabel("Angular position →")
    axes[i, 0].set_ylabel("Radial position →")

    # Mask before refinement — extract .real to get float array
    before = response.mask_responses[i].real
    axes[i, 1].imshow(before, cmap="RdYlGn", aspect="auto", vmin=0, vmax=1)
    axes[i, 1].set_title(f"Filter {i+1}: Mask Before Refinement\n(green=valid, red=masked)")
    axes[i, 1].set_xlabel("Angular position →")

    # Mask after refinement — extract .real
    after = refined.mask_responses[i].real
    axes[i, 2].imshow(after, cmap="RdYlGn", aspect="auto", vmin=0, vmax=1)
    axes[i, 2].set_title(f"Filter {i+1}: Mask After Refinement\n(more red = fragile bits removed)")
    axes[i, 2].set_xlabel("Angular position →")

plt.suptitle("FragileBitRefinement: Removing Unreliable Bits", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### What to look for

The refined mask should have slightly more masked (red) bits than the original, particularly in regions where the iris texture was weak or near boundaries. The number of newly masked fragile bits should be relatively small. A healthy refinement typically masks an additional 2-10% of bits on top of the noise mask.

---

<div style='background-color:honeydew; font-size: 24px;'>
    <center><strong>Encoder</strong></center>
</div>

---


Example from pipeline.yaml:

<code>   - <font color='green'>name: encoder</font>
    algorithm:
      class_name: <font color='blue'>iris.IrisEncoder</font>
      params: {}
    inputs:
      - name: response
        source_node: iris_response_refinement
    callbacks:
      - class_name: <font color='blue'>iris.nodes.validators.object_validators.IsMaskTooSmallValidator</font>
        params:
          min_maskcodes_size: 4096</code>

### What it does  

IrisEncoder takes the complex-valued Gabor filter responses and converts them into the final binary iris code. It does this by looking at the phase of each complex response and assigning a 0 or 1 based on which quadrant of the complex plane the response falls in. This is Daugman's phase quantization - the sign of the real and imaginary parts each contribute one bit, giving two bits per probe point per filter. The result is a compact binary template that can be compared efficiently using the Hamming distance.

**Key Parameters** 

None - IrisEncoder takes no configurable parameters. The encoding logic is fixed: sign of real part -> bit 1, sign of imaginary part -> bit 2. The validator callback IsMaskTooSmallValidator has one parameter:

| Parameter | Input Type | Default Value | What it controls |
|---|---|---|---|
| `min_maskcodes_size` | `int` | 4096 | Minimum number of unmasked bits required - below this, too much of the iris is occluded to match reliably |

**Code & Visualization Cells**   



In [ ]:
from iris.nodes.encoder.iris_encoder import IrisEncoder

refined = pipeline.call_trace.get("iris_response_refinement")

node = IrisEncoder()
template = node.run(response=refined)

print(type(template))
print("Number of iris codes:", len(template.iris_codes))
print("Iris code shape:", template.iris_codes[0].shape)
print("Mask code shape:", template.mask_codes[0].shape)
print("Iris code dtype:", template.iris_codes[0].dtype)  # bool

# Count usable bits
total_bits = template.iris_codes[0].size
unmasked_bits = (~template.mask_codes[0]).sum()
print(f"\nTotal bits per filter: {total_bits}")
print(f"Unmasked (usable) bits: {unmasked_bits}")
print(f"Usable percentage: {100 * unmasked_bits / total_bits:.1f}%")
print(f"Passes min_maskcodes_size check (>= 4096): {unmasked_bits >= 4096}")

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(14, 14))

labels = [
    "Filter 1 - Real phase bit",
    "Filter 1 - Imaginary phase bit",
    "Filter 2 - Real phase bit",
    "Filter 2 - Imaginary phase bit",
]

for filter_idx in range(2):
    for bit_idx in range(2):
        row = filter_idx * 2 + bit_idx

        iris_code = template.iris_codes[filter_idx][:, :, bit_idx].astype(float)
        mask_code = template.mask_codes[filter_idx][:, :, bit_idx].astype(float)

        # Left: raw iris code bits
        axes[row, 0].imshow(iris_code, cmap="binary", aspect="auto")
        axes[row, 0].set_title(f"{labels[row]}\n(black=0, white=1)")
        axes[row, 0].set_xlabel("Angular position →")
        axes[row, 0].set_ylabel("Radial position →")

        # Right: bits with mask overlay
        # green = valid unmasked bit, dark = masked out
        display = np.zeros((*iris_code.shape, 3), dtype=np.float32)
        display[..., 0] = iris_code * (1 - mask_code)   # red where bit=1 and valid
        display[..., 1] = 1 - mask_code                  # green where valid
        display[..., 2] = iris_code * (1 - mask_code)   # blue where bit=1 and valid
        axes[row, 1].imshow(display, aspect="auto")
        axes[row, 1].set_title(f"{labels[row]} + Mask\n(green=valid, black=masked)")
        axes[row, 1].set_xlabel("Angular position →")

plt.suptitle("IrisEncoder: Binary Iris Code (4 bit planes)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# Bit distribution stats — should be ~50/50 for reliable encoding
print("Bit distribution per plane (good iris code = ~50% ones):")
for filter_idx in range(2):
    for bit_idx in range(2):
        code = template.iris_codes[filter_idx][:, :, bit_idx]
        mask = template.mask_codes[filter_idx][:, :, bit_idx]
        
        valid_bits = code[~mask]  # only look at unmasked bits
        ones = valid_bits.sum()
        total = valid_bits.size
        
        print(f"  {labels[filter_idx * 2 + bit_idx]}: "
              f"{ones}/{total} ones ({100*ones/total:.1f}%) — "
              f"{mask.sum()} bits masked")

# Total usable bits across all planes
total_unmasked = sum(
    (~template.mask_codes[f][:, :, b]).sum()
    for f in range(2) for b in range(2)
)
print(f"\nTotal usable bits across all 4 planes: {total_unmasked}")
print(f"Passes min_maskcodes_size (>= 4096): {total_unmasked >= 4096}")

### What to look for 

A good iris code should have roughly 50% 0s and 50% 1s. If the distribution is heavily skewed, something went wrong in the encoding. The mask should cover relatively little of the code - ideally less than 20%. The bit pattern should look visually random with no large uniform blocks, which would indicate a region of corrupted or missing texture.

---

<div style='background-color:honeydew; font-size: 24px;'>
    <center><strong>Bounding Box Estimation</strong></center>
</div>

---


Example from pipeline.yaml:

<code>  - <font color='green'>name: bounding_box_estimation</font>
    algorithm:
      class_name: <font color='blue'>iris.IrisBBoxCalculator</font>
      params: {}
    inputs:
      - name: ir_image
        source_node: input
      - name: geometry_polygons
        source_node: geometry_estimation
    callbacks:</code>

### What it does

It computes the axis-aligned bounding box of the iris region in the original image coordinate space, using the fitted iris geometry polygon. The result is useful for downstream applications that need to crop, display or log which part of the image contained the iris.

**Key Parameters** 

None - fully determined by the iris geometry, no configurable parameters.

**Code & Visualization Cells**   


In [ ]:
from iris.nodes.eye_properties_estimation.iris_bbox_calculator import IrisBBoxCalculator

geometry = pipeline.call_trace.get("geometry_estimation")
bbox = pipeline.call_trace.get("bounding_box_estimation")

# Or run it manually
node = IrisBBoxCalculator()
bbox = node.run(ir_image=ir_image, geometry_polygons=geometry)

print(type(bbox))
print(bbox)

In [ ]:
import matplotlib.patches as patches

fig, ax = plt.subplots(1, 1, figsize=(8, 8))

ax.imshow(img_pixels, cmap="gray")

# Draw the bounding box
# bbox has x_min, y_min, x_max, y_max or similar fields
print(vars(bbox))

rect = patches.Rectangle(
    (bbox.x_min, bbox.y_min),
    bbox.x_max - bbox.x_min,
    bbox.y_max - bbox.y_min,
    linewidth=2, edgecolor="lime", facecolor="none",
    label="Iris bounding box"
)
ax.add_patch(rect)

# Also overlay the iris contour for reference
iris_pts = geometry.iris_array
ax.plot(iris_pts[:, 0], iris_pts[:, 1], "cyan", linewidth=1.5, label="Iris boundary")

ax.legend()
ax.set_title("IrisBBoxCalculator Output")
ax.axis("off")
plt.tight_layout()
plt.show()

### What to look for

The bounding box should tightly enclose the iris contour with minimal padding. If it's dramatically larger than the iris or offset from it, the geometry estimation upstream likely had an error.